In [8]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [9]:

import numpy as np
import matplotlib.pyplot as plt
import h5py

# from lfads_torch.metrics import r2_score
from paper.plot_helpers import class_accuracy

def eval_model(data,predictions,mask=None):
  if mask is None:
    mask = np.ones(data['valid_behavior'][:].shape[0],dtype=bool)

  directions = data['valid_target_direction']
  unique_dirs = sorted(set(directions))

  R2 = np.empty((len(unique_dirs))) # directions
  for i,d in enumerate(unique_dirs):
    precision_error = (predictions[mask & (directions==d)] - data['valid_behavior'][:][mask & (directions==d)])**2
    total_variance = (data['valid_behavior'][:][mask & (directions==d)] - data['valid_behavior'][:][mask & (directions==d)].mean(0))**2+1e-6
    R2[i] = 1 - precision_error.sum()/total_variance.sum() # sum over trials and time and space
  R_UIVE = R2.mean() #uninstructed variance explained

  precision_error = (predictions[mask] - data['valid_behavior'][:][mask])**2
  total_variance = (data['valid_behavior'][:][mask] - data['valid_behavior'][:][mask].mean(0))**2
  vec_R2 = 1 - precision_error.sum()/total_variance.sum() # sum over trials and time and space
  return vec_R2,R_UIVE

In [10]:
tT = 200 # ms target oscillatory frequency
dt = 100 # ms range
oscillatory_criterion = lambda a: (a > 2 * 10 * np.pi/(tT+dt)) & (a < 2 * 10 * np.pi/(tT-dt))


def plot_eigenspectrum(A, ax=None):

    if ax is None:
        _, ax = plt.subplots(1,1)

    evals, _ = np.linalg.eig(A)

    ax.plot([-1,1],[0,0],'k')
    ax.plot([0,0],[-1,1],'k')
    eigen_colors = []
    cc = 0 # color counter
    for e in evals:
        if e.imag >= 0: # plot conjugate pairs in the same color
            ax.plot(e.real,e.imag,'o',color='C'+str(cc))
            eigen_colors.append('C'+str(cc))
            if e.imag > 0:
                ax.plot(e.real,-e.imag,'o',color='C'+str(cc))
                eigen_colors.append('C'+str(cc))
            cc += 1
    # draw a line at 18 degrees
    for t in [tT-dt, tT, tT+dt]:
        tan = np.tan(2 * 10 * np.pi/t)
        style = 'k--' if t == tT else 'r--'
        ax.plot([0,np.sqrt(1-tan**2)],[0,tan],style)
        ax.plot([0,np.sqrt(1-tan**2)],[0,-tan],style)
    # draw a unit circle
    x = np.linspace(-1,1,100)
    y = np.sqrt(1-x**2)
    ax.plot(x,y,'k')
    y = -np.sqrt(1-x**2)
    ax.plot(x,y,'k')

    ax.axis('equal')
    # remove frame and ticks
    ax.axis('off')
    return eigen_colors

def plot_model_summary(model):
    _, ax = plt.subplots(1,3,figsize=(8,3))
    eigen_colors = plot_eigenspectrum(model.A, ax=ax[0])
    _, evecs = np.linalg.eig(model.A)
    ax[1].imshow(model.A, vmin=0, vmax=1)
    m = ax[2].imshow(evecs.real, vmin=0, vmax=1)
    n = model.A.shape[0]
    plt.xticks(np.arange(n),['⚫' for _ in range(n)])
    plt.yticks(np.arange(n),['⚫' for _ in range(n)])

    ax[0].set_title('Eigenvalues')
    ax[1].set_title('A matrix')
    ax[2].set_title('Eigenvectors')

    ax = plt.gca()
    for t,ec in zip(ax.xaxis.get_ticklabels(),eigen_colors):
        t.set_color(ec)
    for t,ec in zip(ax.yaxis.get_ticklabels(),eigen_colors):
        t.set_color(ec) 
    # plt.colorbar(mappable=m)

In [11]:
data = h5py.File("../datasets/Chewie_CO_FF_2016-10-07_session_vel_M1_spikes_go.h5", "r")
print(data.keys())
print(data["valid_recon_data"].shape)

<KeysViewHDF5 ['train_behavior', 'train_encod_data', 'train_epoch', 'train_inds', 'train_pos', 'train_recon_data', 'train_target_direction', 'train_vel', 'valid_behavior', 'valid_encod_data', 'valid_epoch', 'valid_inds', 'valid_pos', 'valid_recon_data', 'valid_target_direction', 'valid_vel']>
(116, 101, 70)


In [12]:
from DPAD import DPADModel
from DPAD.tools.flexible import fitDPADWithFlexibleNonlinearity
idSys = DPADModel()

In [13]:
n_factors = 20
n_beh_factors = 9
mask = np.ones_like(data['train_epoch'],dtype=bool) # train on all epochs
mask = data['train_epoch'][:] == 0 # train on the BL epoch only

yTrain = data["train_recon_data"][mask].reshape((-1,data["train_recon_data"][mask].shape[-1]))
zTrain = data["train_behavior"][mask].reshape((-1,data["train_behavior"][mask].shape[-1]))
nx = n_factors
n1 = n_beh_factors

yTest = data["valid_recon_data"][:].reshape((-1,data["valid_recon_data"][:].shape[-1]))

In [14]:
methodCode = "DPAD_GSUT_iCVF4_RTR2_uAKCzCy1HL64U_ErSV16" # Defines search space over nonlinearities by flexible DPAD
saveDir = "../results/DPAD/" # Directory to save results of fitting DPAD with different nonlinearities
selectedMethodCode, iCVRes = fitDPADWithFlexibleNonlinearity(yTrain, Z=zTrain, nx=nx, n1=n1, settings={"min_cores_to_enable_parallelization": 10}, 
                                                             methodCode=methodCode, saveDir=saveDir) # Fits DPAD with desired nonlinearities and selects the best performing option within training data

idSysF = DPADModel()
args = DPADModel.prepare_args(selectedMethodCode) # Get arguments needed for fitting DPAD with selected configuration
idSysF.fit(yTrain.T, Z=zTrain.T, nx=nx, n1=n1, epochs=2500, **args) # Fit final model

zTestPredF, yTestPredF, xTestPredF = idSysF.predict(yTest) # Run inference to generate predictions

Considering 16 possible combinations of location/type for nonlinearities (this might take a while)...


ValueError: Received an invalid value for `units`, expected a positive integer. Received: units=[64]

NameError: name 'selectedMethodCode' is not defined